# Diabetes prediction

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/iammustafatz/diabetes-prediction-dataset/diabetes_prediction_dataset.csv')
df.head()

## Basic checks

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df['diabetes'].value_counts()

## Cleaning

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

df.shape

## Prepare the data

In [ ]:
# Convert text columns into numeric columns
df = pd.get_dummies(
    df,
    columns=['gender', 'smoking_history'],
    drop_first=True
)

df.head()

In [ ]:
X = df.drop('diabetes', axis=1)
y = df['diabetes']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Logistic Regression

In [ ]:
log_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

log_model.fit(X_train_scaled, y_train)

log_pred = log_model.predict(X_test_scaled)
log_prob = log_model.predict_proba(X_test_scaled)[:, 1]

## Decision Tree

In [ ]:
tree_model = DecisionTreeClassifier(
    max_depth=8,
    random_state=42
)

tree_model.fit(X_train_scaled, y_train)

tree_pred = tree_model.predict(X_test_scaled)
tree_prob = tree_model.predict_proba(X_test_scaled)[:, 1]

## Random Forest

In [ ]:
forest_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

forest_model.fit(X_train_scaled, y_train)

forest_pred = forest_model.predict(X_test_scaled)
forest_prob = forest_model.predict_proba(X_test_scaled)[:, 1]

## Compare the models

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, log_pred),
        accuracy_score(y_test, tree_pred),
        accuracy_score(y_test, forest_pred)
    ],
    'Precision': [
        precision_score(y_test, log_pred),
        precision_score(y_test, tree_pred),
        precision_score(y_test, forest_pred)
    ],
    'Recall': [
        recall_score(y_test, log_pred),
        recall_score(y_test, tree_pred),
        recall_score(y_test, forest_pred)
    ],
    'F1-score': [
        f1_score(y_test, log_pred),
        f1_score(y_test, tree_pred),
        f1_score(y_test, forest_pred)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, log_prob),
        roc_auc_score(y_test, tree_prob),
        roc_auc_score(y_test, forest_prob)
    ]
})

results.round(4)

## Best model

In [ ]:
# F1-score balances precision and recall
best_model_name = results.loc[results['F1-score'].idxmax(), 'Model']

predictions = {
    'Logistic Regression': log_pred,
    'Decision Tree': tree_pred,
    'Random Forest': forest_pred
}

best_pred = predictions[best_model_name]
best_model_name

In [ ]:
cm = confusion_matrix(y_test, best_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - ' + best_model_name)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Conclusion

In [ ]:
best_result = results.loc[results['F1-score'].idxmax()]

print(f"{best_model_name} is the best model because it has the highest F1-score of {best_result['F1-score']:.4f}.")
print(f"It has an accuracy of {best_result['Accuracy']:.4f}, precision of {best_result['Precision']:.4f}, recall of {best_result['Recall']:.4f}, and ROC-AUC of {best_result['ROC-AUC']:.4f}.")
print("F1-score is used for selection because the diabetes classes are imbalanced, so accuracy alone can be misleading.")